# Phishing URL Detection Using Machine Learning
**Student:** Rashmi Nagarkoti  
**Programme:** BCA (AI & Data Science)  
**Bootcamp:** IBM SkillsBuild Cyber Security BootCamp 2026 — BharatCares

## Project overview
This notebook builds a baseline machine-learning classifier that estimates whether a URL is phishing or legitimate from the URL text itself. It uses character-level TF–IDF features and Logistic Regression, then evaluates the model on a held-out test set.

**Important:** This is an educational prototype, not a production security tool. A prediction is not proof that a website is safe or malicious. Do not open suspicious URLs to test them.

## 1. Install / import dependencies
If any import fails, install the packages listed in `requirements.txt` and restart the kernel.

In [ ]:
import re
import sys
import warnings
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, ConfusionMatrixDisplay, confusion_matrix
)

RANDOM_STATE = 42
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)

## 2. Load the public dataset
The notebook downloads a CSV from the dataset repository. An internet connection is required for this cell. The repository documents a two-column dataset: `URL` and `Label`, where `1` means legitimate and `0` means phishing.

In [ ]:
DATASET_URL = "https://raw.githubusercontent.com/RajaMuhammadAwais/phishing-url-dataset/master/phishing_url_dataset.csv"

try:
    df = pd.read_csv(DATASET_URL)
except Exception as exc:
    raise RuntimeError(
        "Could not download the dataset. Check your internet connection, "
        "or download phishing_url_dataset.csv from the README dataset link "
        "and replace DATASET_URL with the local file path."
    ) from exc

print("Original shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

## 3. Clean and inspect the data
The pipeline checks the expected columns, removes missing values and duplicate URLs, and validates the binary labels. A reproducible sample is used to keep training manageable on a student laptop. Increase `MAX_ROWS` if your computer has sufficient memory.

In [ ]:
# Normalize column names in a tolerant way
column_lookup = {str(c).strip().lower(): c for c in df.columns}
url_col = column_lookup.get("url")
label_col = column_lookup.get("label")

if url_col is None or label_col is None:
    raise ValueError(
        "Expected columns named URL and Label. "
        f"Found columns: {df.columns.tolist()}"
    )

data = df[[url_col, label_col]].copy()
data.columns = ["url", "label"]
data["url"] = data["url"].astype("string").str.strip()
data["label"] = pd.to_numeric(data["label"], errors="coerce")

data = data.dropna(subset=["url", "label"])
data = data[data["url"].str.len() > 0]
data = data[data["label"].isin([0, 1])]
data["label"] = data["label"].astype(int)
data = data.drop_duplicates(subset=["url"]).reset_index(drop=True)

print("Cleaned shape:", data.shape)
print("Label counts (0 = phishing, 1 = legitimate):")
display(data["label"].value_counts().sort_index().rename(index={0:"Phishing (0)", 1:"Legitimate (1)"}))

if data["label"].nunique() != 2:
    raise ValueError("The cleaned dataset must contain both label classes.")

MAX_ROWS = 40000
if len(data) > MAX_ROWS:
    data = data.groupby("label", group_keys=False).apply(
        lambda part: part.sample(
            n=min(len(part), MAX_ROWS // 2),
            random_state=RANDOM_STATE
        )
    ).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print("Rows used for this run:", len(data))
display(data.head())

## 4. Explore the dataset
The chart shows the class balance in the data used for modelling.

In [ ]:
label_names = data["label"].map({0: "Phishing", 1: "Legitimate"})
label_names.value_counts().reindex(["Phishing", "Legitimate"]).plot(
    kind="bar", rot=0, title="Class distribution in modelling data"
)
plt.xlabel("URL class")
plt.ylabel("Number of URLs")
plt.tight_layout()
plt.show()

data["url_length"] = data["url"].str.len()
display(data.groupby("label")["url_length"].describe().rename(
    index={0: "Phishing (0)", 1: "Legitimate (1)"}
))

## 5. Split the data and train the model
Character n-grams can capture URL patterns such as unusual separators, strings and spelling patterns without opening or visiting the URL. The split is stratified so both classes are represented in train and test sets.

In [ ]:
X = data["url"].astype(str)
y = data["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

model = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="char",
        ngram_range=(3, 5),
        min_df=2,
        max_features=150000,
        sublinear_tf=True,
        dtype=np.float32
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

model.fit(X_train, y_train)
print(f"Training rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")
print("Training completed.")

## 6. Evaluate the model
Accuracy alone can hide errors, so precision, recall, F1-score and the confusion matrix are also reported. Record the actual values produced when you run the notebook; results can vary with dataset changes.

In [ ]:
y_pred = model.predict(X_test)

metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision (phishing)", "Recall (phishing)", "F1-score (phishing)"],
    "Score": [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred, pos_label=0, zero_division=0),
        recall_score(y_test, y_pred, pos_label=0, zero_division=0),
        f1_score(y_test, y_pred, pos_label=0, zero_division=0)
    ]
})
display(metrics.style.format({"Score": "{:.4f}"}))

print(classification_report(
    y_test, y_pred,
    labels=[0, 1],
    target_names=["Phishing (0)", "Legitimate (1)"],
    zero_division=0
))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    labels=[0, 1],
    display_labels=["Phishing", "Legitimate"],
    cmap="Blues",
    values_format="d"
)
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

## 7. Try a URL string
The examples below are illustrative strings only. The model analyses the text and does not visit the websites. Predictions are estimates and must not be treated as a safety guarantee.

In [ ]:
def predict_url(url_text: str) -> dict:
    """Classify URL text without visiting the URL."""
    if not isinstance(url_text, str) or not url_text.strip():
        raise ValueError("Enter a non-empty URL string.")

    prediction = int(model.predict([url_text.strip()])[0])
    probabilities = model.predict_proba([url_text.strip()])[0]
    class_to_probability = dict(zip(model.named_steps["classifier"].classes_, probabilities))

    return {
        "input_url": url_text.strip(),
        "prediction": "Likely legitimate" if prediction == 1 else "Potential phishing",
        "phishing_score": round(float(class_to_probability.get(0, np.nan)), 4),
        "legitimate_score": round(float(class_to_probability.get(1, np.nan)), 4),
        "warning": "Model estimate only; do not use as the sole security decision."
    }

# Replace this with a URL string you already have; do not open suspicious links.
example_url = "https://www.example.com/account/login"
predict_url(example_url)

## 8. Findings, limitations and future improvements

### Findings
Use the metric table and confusion matrix generated above to describe the model's measured performance. Do not copy assumed or example scores into the report.

### Limitations
- The model learns patterns from the supplied labelled dataset; it can be wrong on new or previously unseen URLs.
- Dataset labels may contain errors or reflect the time and sources from which they were collected.
- The model uses URL text only. It does not inspect page content, certificates, DNS, redirects or reputation feeds.
- A legitimate-looking URL may still be dangerous, and a suspicious-looking URL may be legitimate.
- No live website is opened by this notebook.

### Future improvements
- Evaluate on a separate, newer dataset from a different source.
- Compare Logistic Regression with Linear SVM and tree-based models using the same split.
- Add careful URL/domain features and explainability analysis.
- Track false negatives and false positives, and assess performance by URL source and collection date.

## 9. Conclusion
This project demonstrates an end-to-end supervised-learning workflow for a cybersecurity use case: dataset loading, cleaning, exploratory analysis, feature extraction, model training, evaluation and a reusable prediction function. It is a learning prototype and should not replace browser protections, security software or expert review.

## References
- Dataset repository: https://github.com/RajaMuhammadAwais/phishing-url-dataset
- scikit-learn documentation: https://scikit-learn.org/stable/
- pandas documentation: https://pandas.pydata.org/docs/
